# 🎓 Student Performance Predictor: EDA & Machine Learning Workflow

**Author:** Ayush Panda  
**Degree:** B.Tech Artificial Intelligence  
**Institution:** SRM Institute of Science and Technology, Chennai  

---

## 📖 Project Overview
This notebook demonstrates a complete, beginner-friendly **Supervised Machine Learning** pipeline to predict a student's final academic score based on academic and lifestyle factors:
- **Study Hours per Day**
- **Attendance Percentage**
- **Assignment / Test Score**
- **Sleep Hours per Day**

We use **Linear Regression** as our primary algorithm to model the linear relationship between these features and the final score.

> **Note on Dataset:** The dataset used in this project is synthetic and designed strictly for educational and demonstration purposes. It does not contain any private or real student records.

## 1. Import Necessary Libraries
We begin by importing our core data science libraries:
- `pandas` for tabular data manipulation
- `numpy` for numerical operations
- `matplotlib.pyplot` and `seaborn` for exploratory data visualization
- `sklearn` modules for model training and performance evaluation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styling for clean, professional charts
sns.set_theme(style="whitegrid", palette="deep")
print("Libraries imported successfully!")

## 2. Load and Inspect the Dataset
We load `student_performance.csv` from the `data/` directory and inspect its structure, dimensions, and first few rows.

In [ ]:
# Load dataset from the data folder
data_path = os.path.join("..", "data", "student_performance.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "student_performance.csv")

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Data Hygiene Checks: Checking for Missing Values and Data Types
Machine learning models cannot process missing (NaN) values without imputation or dropping. Let's verify our data.

In [ ]:
# Check for null values across all columns
print("Missing values per column:")
print(df.isnull().sum())

# Display column types and non-null counts
print("\nData Types and Info:")
df.info()

## 3. Summary Statistics
Examining the minimum, maximum, mean, and quartiles helps us understand the spread and scale of each feature.

In [ ]:
df.describe()

## 4. Exploratory Data Analysis (EDA)
EDA helps us visually inspect patterns, test assumptions, and identify correlations before feeding data into our model.

### 4.1 Distribution of the Target Variable (`final_score`)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['final_score'], kde=True, color='#9467bd', bins=25, edgecolor='black')
plt.axvline(df['final_score'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['final_score'].mean():.2f}")
plt.axvline(df['final_score'].median(), color='green', linestyle=':', linewidth=2, label=f"Median: {df['final_score'].median():.2f}")
plt.title("Distribution of Final Scores", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Final Score (out of 100)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

### 4.2 Feature Relationships with Final Score
Let's plot scatter plots with linear regression trendlines for each feature against the target.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Study Hours vs Score
sns.regplot(data=df, x='study_hours_per_day', y='final_score', ax=axes[0],
            scatter_kws={'alpha': 0.5, 'color': '#1f77b4'}, line_kws={'color': '#d62728'})
axes[0].set_title("Study Hours vs Final Score", fontweight='bold')
axes[0].set_xlabel("Study Hours per Day")
axes[0].set_ylabel("Final Score")

# Attendance vs Score
sns.regplot(data=df, x='attendance_percent', y='final_score', ax=axes[1],
            scatter_kws={'alpha': 0.5, 'color': '#2ca02c'}, line_kws={'color': '#d62728'})
axes[1].set_title("Attendance % vs Final Score", fontweight='bold')
axes[1].set_xlabel("Attendance Percentage (%)")
axes[1].set_ylabel("Final Score")

# Assignment Score vs Final Score
sns.regplot(data=df, x='assignment_score', y='final_score', ax=axes[2],
            scatter_kws={'alpha': 0.5, 'color': '#ff7f0e'}, line_kws={'color': '#d62728'})
axes[2].set_title("Assignment Score vs Final Score", fontweight='bold')
axes[2].set_xlabel("Assignment Score")
axes[2].set_ylabel("Final Score")

plt.tight_layout()
plt.show()

### 4.3 Correlation Heatmap
The Pearson correlation coefficient measures the strength and direction of linear association between variables, ranging from -1.0 to +1.0.

In [ ]:
plt.figure(figsize=(8, 6))
correlation_matrix = df.corr()
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Correlation Heatmap of Student Features", fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 5. Machine Learning Pipeline

### 5.1 Define Features ($X$) and Target ($y$)
- **Features ($X$):** Independent variables used to make predictions.
- **Target ($y$):** Dependent variable we want to predict (`final_score`).

In [ ]:
features = ['study_hours_per_day', 'attendance_percent', 'assignment_score', 'sleep_hours']
target = 'final_score'

X = df[features]
y = df[target]

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")

### 5.2 Train-Test Split
We split our data into:
- **Training Set (80%):** Used by the model to learn coefficients.
- **Testing Set (20%):** Kept unseen to evaluate generalization performance.
We set `random_state=42` to ensure reproducible results across runs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

### 5.3 Train Linear Regression Model
The linear regression model assumes:
$$\hat{y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4$$
where:
- $\beta_0$ is the intercept
- $\beta_1, \beta_2, \dots$ are feature coefficients (slopes)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Intercept (beta_0): {model.intercept_:.4f}")
print("Coefficients:")
for feat, coef in zip(features, model.coef_):
    print(f"  - {feat:22s}: +{coef:.4f}")

## 6. Model Evaluation
We evaluate the model on the unseen test set using three standard metrics:
1. **Mean Absolute Error (MAE):** The average magnitude of errors in the same units as the score.
2. **Mean Squared Error (MSE):** The average of squared errors, penalizing larger deviations more heavily.
3. **R-squared ($R^2$ Score):** The proportion of variance in the final score that is explained by the input features.

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 45)
print("       MODEL EVALUATION METRICS")
print("=" * 45)
print(f"Mean Absolute Error (MAE) : {mae:.4f}")
print(f"Mean Squared Error (MSE)  : {mse:.4f}")
print(f"Root Mean Squared Error   : {rmse:.4f}")
print(f"R-squared (R2 Score)      : {r2:.4f} ({r2 * 100:.2f}% variance explained)")
print("=" * 45)

### 6.1 Actual vs. Predicted Plot
In an ideal model, points lie exactly on the dashed red line ($y = x$).

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='#17becf', edgecolors='k', linewidth=0.5, label='Predicted vs Actual')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal Fit (y = x)')
plt.title("Actual vs. Predicted Scores on Test Set", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Actual Final Score")
plt.ylabel("Predicted Final Score")
plt.legend()
plt.tight_layout()
plt.show()

### 6.2 Residual Analysis
Residuals are the differences between actual and predicted values: $\text{Residual} = y - \hat{y}$.
A good linear regression model exhibits residuals distributed normally around 0.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='#e377c2', bins=20, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
plt.title("Residual Distribution (Errors)", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Sample Prediction Demonstration
Let's test our trained model on a sample student profile:

In [ ]:
sample_student = pd.DataFrame([{
    'study_hours_per_day': 6.5,
    'attendance_percent': 90.0,
    'assignment_score': 85.0,
    'sleep_hours': 7.5
}])

predicted_score = model.predict(sample_student)[0]
clamped_score = max(0.0, min(100.0, predicted_score))

print("Sample Student Input:")
display(sample_student)
print(f"\nPredicted Final Score: {clamped_score:.2f} / 100")

## 8. Summary & Key Findings
- **Study Hours** and **Assignment Scores** are the strongest positive contributors to final exam outcomes.
- **Attendance** provides a steady baseline support to academic success.
- **Sleep Hours** contribute positively to overall focus and performance.
- Linear Regression provides high interpretability, allowing us to explain precisely how each individual factor shifts expected final scores.